In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.Silver_Helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *

In [0]:
result_df  = spark.read.table(bronze_table).filter(col("batch_id")==v_batch_id)

In [0]:
result_drop_df = result_df.drop('url')

In [0]:
result_renamed_df = (
    result_drop_df
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "raceId":"race_id",
        "driverId":"driver_id",
        "raceName":"race_name",
        "positionText":"finish_position_text",
        "date":"race_date",
        "grid":"grid_position",
        "laps":"completed_laps",
        "number":"car_number",
        "position":"finsh_position"
    })
)

In [0]:
result_null_df =(
    result_renamed_df
    .filter(col("season").isNotNull() & col("round").isNotNull() & col("driver_id").isNotNull() & col("constructor_id").isNotNull())
) 

In [0]:
result_distincts_df = result_null_df.dropDuplicates(['season','round','driver_id','constructor_id'])

In [0]:
result_final_df = (
    result_distincts_df
    .withColumn("race_name",initcap(col("race_name")))
)

In [0]:
columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finsh_position",
        "finish_position_text",
        "status",
        "ingestion_timestamp",
        "source_files",
        "batch_id"
    ]

In [0]:
write_to_silver(input_df=result_final_df,target_table=silver_table,
                merge_condn="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.table(silver_table))